<a href="https://colab.research.google.com/github/priscilla-stn/desafios_tecnicos/blob/main/Desafio_1/Desafio1_API_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Instala as bibliotecas usadas no guia.
!pip install -q gspread faker

In [3]:
# Importa as ferramentas para autenticação, geração de dados e sorteios.
import json, gspread
from google.colab import userdata
from faker import Faker
import random

# Lê a credencial salva nos Secrets e conecta à planilha.
info = json.loads(userdata.get("GCP_SA_JSON"))
gc = gspread.service_account_from_dict(info)
# Exibe o e-mail da conta autenticada.
print("Conectado como:", info["client_email"])

Conectado como: desafio1-python@zippy-chain-513817-i9.iam.gserviceaccount.com


In [4]:
# Informe o ID da planilha e o nome que será dado à aba.
SHEET_ID = "1a8QS7bKAM-vhjpaCrRxOVfQcFI_l1YmRTCmCWFCnZ3w"
NOME_TABELA = "clientes"   # nome da tabela na ficha do tema, ex.: "vagas"

# Abre a planilha e seleciona sua primeira aba.
sh = gc.open_by_key(SHEET_ID)
ws = sh.sheet1
# Renomeia a aba e exibe os dados da planilha.
ws.update_title(NOME_TABELA)     # a aba passa a ter o nome da tabela
print(sh.title, "|", ws.title, "|", sh.url)

Desafio_01_Atendimento_Cliente | clientes | https://docs.google.com/spreadsheets/d/1a8QS7bKAM-vhjpaCrRxOVfQcFI_l1YmRTCmCWFCnZ3w


In [10]:
# Substitua pelos nomes EXATOS de coluna da ficha do seu tema
# Define as colunas que formarão o cabeçalho da tabela.
COLUNAS = ["cliente_id", "segmento", "canal_preferido", "data_cadastro"]

# Escreve o cabeçalho na primeira célula e confere o resultado.
ws.update(range_name="A1", values=[COLUNAS])
print("Cabeçalho confere:", ws.row_values(1) == COLUNAS)

Cabeçalho confere: True


In [13]:
# Configura os geradores aleatórios para produzir resultados reproduzíveis.
fake = Faker("pt_BR")
Faker.seed(42)
random.seed(42)

def gerar_linha(i):
    # Cria um registro fictício para as colunas definidas no esquema.
    # Personalize de acordo com o seu esquema de dados
    return {
        "cliente_id": i,
        "segmento": random.choice(["pessoa_fisica", "pequena_empresa", "grande_empresa"]),
        "canal_preferido": random.choice(["telefone", "chat", "email", "presencial"]),
        "data_cadastro": fake.date_this_year().isoformat(),   # datas como texto
    }

# Gera 10 registros numerados de 1 a 10.
linhas = [gerar_linha(i) for i in range(1, 11)]

# Organiza os valores na ordem do cabeçalho; nome de coluna incorreto gera KeyError.
valores = [[linha[c] for c in COLUNAS] for linha in linhas]
# Acrescenta os registros à planilha e informa quantos foram enviados.
ws.append_rows(valores, value_input_option="RAW")
print(len(valores), "linhas enviadas")

10 linhas enviadas


In [14]:
# Lê todos os valores da aba para conferir o conteúdo enviado.
valores_lidos = ws.get_all_values()
# Exibe a quantidade de linhas, confere o cabeçalho e mostra exemplos.
print("Linhas (cabeçalho + dados):", len(valores_lidos))    # esperado: 11
print("Cabeçalho confere:", valores_lidos[0] == COLUNAS)
print(valores_lidos[:3])

Linhas (cabeçalho + dados): 11
Cabeçalho confere: True
[['cliente_id', 'segmento', 'canal_preferido', 'data_cadastro'], ['1', 'grande_empresa', 'telefone', '2026-07-16'], ['2', 'pessoa_fisica', 'email', '2026-01-08']]
